In [5]:
import sys, os
from pathlib import Path

def add_repo_path():
    """
    stock_forecast 프로젝트 루트를 자동 탐색하고,
    해당 경로를 sys.path에 추가하여 import 오류를 방지합니다.
    """
    # __file__이 정의된 경우 (일반 .py 파일)
    if '__file__' in globals():
        current = Path(__file__).resolve().parent
    else:
        # Jupyter Notebook이나 대화형 환경
        current = Path.cwd()

    # 현재 디렉토리와 상위 디렉토리들을 탐색
    for parent in [current] + list(current.parents):
        # DATA 폴더가 존재하는 경로를 찾으면 sys.path에 추가
        if (parent / "DATA").exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            print(f"[INFO] Project root added to sys.path: {parent}")
            return str(parent)

    # 만약 못 찾을 경우 대비 - fallback 경로 지정
    fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
    if os.path.isdir(fallback):
        if fallback not in sys.path:
            sys.path.insert(0, fallback)
        print(f"[WARNING] Using fallback path: {fallback}")
        return fallback

    raise FileNotFoundError("❌ DATA 폴더를 찾을 수 없습니다.")

# 경로 추가 실행
project_root = add_repo_path()

# korea_fs_loader.py

# -------------------------------------------------------------------
# 0. 환경설정
# -------------------------------------------------------------------
import os
import io
import re
import zipfile
import time
import datetime as dt
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from sqlalchemy import create_engine, text
from typing import List, Dict, Optional


DART_FS_URL = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
REPRT_CODES_DEFAULT = ["11011"]   # 연간 보고서


# -------------------------------------------------------------------
# 1. 기업코드 관련 함수
# -------------------------------------------------------------------
def load_corp_code(api_key: str, cache_path: str = "dart_corp_codes.csv") -> pd.DataFrame:
    """corp_code 목록 로딩 (캐시 사용)"""
    if os.path.exists(cache_path):
        return pd.read_csv(cache_path, dtype=str)

    # 네트워크 호출
    url = "https://opendart.fss.or.kr/api/corpCode.xml"
    resp = requests.get(url, params={"crtfc_key": api_key}, timeout=30)
    resp.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        xml_name = [name for name in z.namelist() if name.endswith(".xml")][0]
        with z.open(xml_name) as f:
            tree = ET.parse(f)

    root = tree.getroot()
    rows = []
    for elem in root.findall("list"):
        rows.append({
            "corp_code": elem.findtext("corp_code"),
            "corp_name": elem.findtext("corp_name"),
            "stock_code": elem.findtext("stock_code")
        })
    df = pd.DataFrame(rows)
    df = df[df["stock_code"].notna() & (df["stock_code"] != "")]
    df.to_csv(cache_path, index=False, encoding="utf-8-sig")
    return df


def get_corp_info(corp_df: pd.DataFrame, ticker: str) -> Optional[Dict[str, str]]:
    """ticker → corp_code 변환"""
    ticker = str(ticker).zfill(6)
    row = corp_df.loc[corp_df["stock_code"] == ticker]
    if row.empty:
        return None
    r = row.iloc[0]
    return {"corp_code": r["corp_code"], "corp_name": r["corp_name"], "stock_code": ticker}


# -------------------------------------------------------------------
# 2. DART API 호출 함수
# -------------------------------------------------------------------
def fetch_fs(api_key: str, corp_code: str, bsns_year: int, reprt_code: str) -> pd.DataFrame:
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": bsns_year,
        "reprt_code": reprt_code,
        "fs_div": "CFS"
    }
    resp = requests.get(DART_FS_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    if data.get("status") != "000":
        return pd.DataFrame()

    return pd.DataFrame(data.get("list", []))


# -------------------------------------------------------------------
# 3. Wide 변환 + 재무비율 계산
# -------------------------------------------------------------------
def prepare_wide(df_raw: pd.DataFrame) -> pd.DataFrame:
    if df_raw.empty:
        return pd.DataFrame()

    df = df_raw.copy()

    def to_float(x):
        try:
            return float(str(x).replace(",", ""))
        except:
            return None

    df["thstrm_amount"] = df["thstrm_amount"].apply(to_float)

    # 날짜 생성: bsns_year를 기준으로 12월 31일로 설정 (연간 보고서)
    # reprt_code에 따라 다른 날짜를 설정할 수도 있음
    # 11011: 사업보고서 (연간) -> 12-31
    # 11012: 반기보고서 -> 06-30
    # 11013: 1분기보고서 -> 03-31
    # 11014: 3분기보고서 -> 09-30

    def get_report_date(row):
        year = int(row['bsns_year'])
        reprt_code = row['reprt_code']

        if reprt_code == '11011':  # 연간
            return pd.Timestamp(year=year, month=12, day=31)
        elif reprt_code == '11012':  # 반기
            return pd.Timestamp(year=year, month=6, day=30)
        elif reprt_code == '11013':  # 1분기
            return pd.Timestamp(year=year, month=3, day=31)
        elif reprt_code == '11014':  # 3분기
            return pd.Timestamp(year=year, month=9, day=30)
        else:
            return pd.Timestamp(year=year, month=12, day=31)

    df["date"] = df.apply(get_report_date, axis=1)

    # Pivot
    wide = df.pivot_table(
        index=["date", "corp_name", "stock_code"],
        columns="account_nm",
        values="thstrm_amount",
        aggfunc="first"
    ).reset_index()

    return wide


def compute_ratios(wide: pd.DataFrame) -> pd.DataFrame:
    if wide.empty:
        return wide

    def safe_div(a, b):
        if pd.isna(a) or pd.isna(b) or b == 0:
            return None
        return a / b

    # 주요 비율 계산
    wide["GPM"] = wide.apply(lambda row: safe_div(row.get("매출총이익"), row.get("매출액")), axis=1)
    wide["OPM"] = wide.apply(lambda row: safe_div(row.get("영업이익"), row.get("매출액")), axis=1)
    wide["NIM"] = wide.apply(lambda row: safe_div(row.get("당기순이익"), row.get("매출액")), axis=1)
    wide["ROA"] = wide.apply(lambda row: safe_div(row.get("당기순이익"), row.get("자산총계")), axis=1)
    wide["ROE"] = wide.apply(lambda row: safe_div(row.get("당기순이익"), row.get("자본총계")), axis=1)
    wide["Debt_Ratio"] = wide.apply(lambda row: safe_div(row.get("부채총계"), row.get("자본총계")), axis=1)

    return wide


def to_long(wide: pd.DataFrame) -> pd.DataFrame:
    ratio_cols = ["GPM", "OPM", "NIM", "ROA", "ROE", "Debt_Ratio"]
    id_cols = ["date", "corp_name", "stock_code"]

    df_long = wide[id_cols + ratio_cols].melt(
        id_vars=id_cols,
        value_vars=ratio_cols,
        var_name="indicator",
        value_name="value"
    )
    df_long = df_long.dropna(subset=["value"])
    df_long = df_long.rename(columns={"corp_name": "company_name", "stock_code": "ticker"})
    return df_long


# -------------------------------------------------------------------
# 4. 티커 단위 전체 처리 함수 (사용자가 호출)
# -------------------------------------------------------------------
def collect_fs_for_ticker(
    api_key: str,
    corp_df: pd.DataFrame,
    ticker: str,
    start_year: int = 2015,
    end_year: int = None,
    reprt_codes: List[str] = None
) -> pd.DataFrame:

    end_year = end_year or dt.date.today().year
    reprt_codes = reprt_codes or REPRT_CODES_DEFAULT

    info = get_corp_info(corp_df, ticker)
    if info is None:
        print(f"{ticker} not found.")
        return pd.DataFrame()

    corp_code = info["corp_code"]
    corp_name = info["corp_name"]

    all_df = []
    for y in range(start_year, end_year + 1):
        for rc in reprt_codes:
            print(f"Fetching {corp_name} ({ticker}) - Year: {y}, Report: {rc}")
            df = fetch_fs(api_key, corp_code, y, rc)
            if not df.empty:
                df["stock_code"] = ticker
                df["corp_name"] = corp_name
                all_df.append(df)
            time.sleep(0.1)  # API 호출 제한 방지

    if not all_df:
        print(f"No data found for {ticker}")
        return pd.DataFrame()

    raw = pd.concat(all_df, ignore_index=True)
    wide = prepare_wide(raw)
    wide = compute_ratios(wide)
    long_df = to_long(wide)

    print(f"✅ Collected {len(long_df)} records for {corp_name} ({ticker})")
    return long_df


# -------------------------------------------------------------------
# 5. DB 업로드 함수
# -------------------------------------------------------------------
def upload_to_db(df_long: pd.DataFrame, db_info: dict, table_name: str = "korea_fs_data_from_DART"):
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}?charset=utf8mb4"
    )
    df_long.to_sql(table_name, engine, if_exists="append", index=False)
    print(f"DB 업로드 완료: {len(df_long)} rows")

[INFO] Project root added to sys.path: C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast


In [6]:
# from korea_fs_loader import (
#     load_corp_code,
#     collect_fs_for_ticker,
#     upload_to_db
# )

API_KEY = "50424484a46daa88b34fcf875f40ca12b79e1fc1"
corp_df = load_corp_code(API_KEY)

# 삼성전자 데이터 수집
df_samsung = collect_fs_for_ticker(
    api_key=API_KEY,
    corp_df=corp_df,
    ticker="005930"
)


# db_info = {
#     'host': get_db_host(),
#     'port': 3307,
#     'user' : 'stox7412',
#     'password' : 'Apt106503!~',
#     'database': 'investar'
# }

# upload_to_db(df_samsung, db_info)

Fetching 삼성전자 (005930) - Year: 2015, Report: 11011
Fetching 삼성전자 (005930) - Year: 2016, Report: 11011
Fetching 삼성전자 (005930) - Year: 2017, Report: 11011
Fetching 삼성전자 (005930) - Year: 2018, Report: 11011
Fetching 삼성전자 (005930) - Year: 2019, Report: 11011
Fetching 삼성전자 (005930) - Year: 2020, Report: 11011
Fetching 삼성전자 (005930) - Year: 2021, Report: 11011
Fetching 삼성전자 (005930) - Year: 2022, Report: 11011
Fetching 삼성전자 (005930) - Year: 2023, Report: 11011
Fetching 삼성전자 (005930) - Year: 2024, Report: 11011
Fetching 삼성전자 (005930) - Year: 2025, Report: 11011
✅ Collected 31 records for 삼성전자 (005930)


In [7]:
df_samsung

,date,company_name,ticker,indicator,value
9,2024-12-31,삼성전자,005930,GPM,0.379926
19,2024-12-31,삼성전자,005930,OPM,0.108771
29,2024-12-31,삼성전자,005930,NIM,0.114505
30,2015-12-31,삼성전자,005930,ROA,0.078703
31,2016-12-31,삼성전자,005930,ROA,0.086683
32,2017-12-31,삼성전자,005930,ROA,0.139806
33,2018-12-31,삼성전자,005930,ROA,0.130673
34,2019-12-31,삼성전자,005930,ROA,0.061659
35,2020-12-31,삼성전자,005930,ROA,0.069818
36,2021-12-31,삼성전자,005930,ROA,0.093543


In [4]:
# API 응답 구조 확인
API_KEY = "50424484a46daa88b34fcf875f40ca12b79e1fc1"
corp_df = load_corp_code(API_KEY)

# 삼성전자 정보 가져오기
info = get_corp_info(corp_df, "005930")
print(f"기업 정보: {info}")

# 단일 연도 데이터 확인
test_df = fetch_fs(API_KEY, info["corp_code"], 2023, "11011")
print(f"\n컬럼 목록:\n{test_df.columns.tolist()}")
print(f"\n데이터 샘플:\n{test_df.head()}")

기업 정보: {'corp_code': '00126380', 'corp_name': '삼성전자', 'stock_code': '005930'}

컬럼 목록:
['rcept_no', 'reprt_code', 'bsns_year', 'corp_code', 'sj_div', 'sj_nm', 'account_id', 'account_nm', 'account_detail', 'thstrm_nm', 'thstrm_amount', 'frmtrm_nm', 'frmtrm_amount', 'bfefrmtrm_nm', 'bfefrmtrm_amount', 'ord', 'currency', 'thstrm_add_amount']

데이터 샘플:
         rcept_no reprt_code bsns_year corp_code sj_div  sj_nm  \
0  20240312000736      11011      2023  00126380     BS  재무상태표   
1  20240312000736      11011      2023  00126380     BS  재무상태표   
2  20240312000736      11011      2023  00126380     BS  재무상태표   
3  20240312000736      11011      2023  00126380     BS  재무상태표   
4  20240312000736      11011      2023  00126380     BS  재무상태표   

                         account_id account_nm account_detail thstrm_nm  \
0                  ifrs-full_Assets       자산총계              -    제 55 기   
1           ifrs-full_CurrentAssets       유동자산              -    제 55 기   
2    dart_ShortTermOtherRecei